**REFIT VERSION** (final 2D number). For each fold, the number of epochs is taken from the nested run (best epoch on the
12 inner-validation patients); the model is then retrained on **all 81 training patients** for exactly that many epochs,
with the same cosine schedule and **no further selection**. The outer fold is used only for evaluation. This removes both
the selection optimism (held-out run) and the data loss (nested run).

Everything else (data, augmentation, model, decoding, evaluation) is identical to the nested notebook.

**Inputs**: `preprocessed` (with ct.nii.gz), `run1` (split.json), `v3` output (v3_hfvalid_cv.csv, v3_vsd.csv),
VSD output (prepare_log.csv + data/), **the lm2d_nested output** (lm2d_fold0-4.pt, only to read the epoch counts),
optional: plane_errors.csv.
**Accelerator: GPU T4.** About 1-1.5 h. Output: `/kaggle/working/lm2d_refit/` (+ lm2d_refit.zip).


In [ ]:
!pip install -q monai==1.6.1

In [ ]:
import glob, json, time, shutil
from pathlib import Path
import numpy as np, pandas as pd, SimpleITK as sitk, torch, torch.nn.functional as F, matplotlib.pyplot as plt
from scipy import ndimage
from scipy.stats import wilcoxon, binomtest
from torch.utils.data import Dataset, DataLoader
from monai.networks.nets import UNet

def find(pat, required=True):
    hits = glob.glob(f"/kaggle/input/**/{pat}", recursive=True)
    assert hits or not required, f"{pat} not found under /kaggle/input: add the dataset that contains it"
    return Path(hits[0]).parent if hits else None
ROOT, SPLIT, RES = find("dataset.csv"), find("split.json"), find("v3_hfvalid_cv.csv")
VSDIN, PLANES = find("prepare_log.csv", required=False), find("plane_errors.csv", required=False)
NESTED = find("lm2d_fold0.pt")                     # nested run, only for the epoch counts
OUT = Path("/kaggle/working/lm2d_refit"); OUT.mkdir(parents=True, exist_ok=True)
FULL, VOX, DRR_SCALE = (256, 128, 96), 2.0, 6.0
device = "cuda" if torch.cuda.is_available() else "cpu"; assert device == "cuda", "set Accelerator to GPU"
folds = json.load(open(SPLIT / "split.json")); ids = sorted(p for f in folds for p in f)
FOLD = {p: k for k, f in enumerate(folds) for p in f}
rd = lambda p: sitk.GetArrayFromImage(sitk.ReadImage(str(p)))
DRR = {p: [rd(ROOT / p / f"{v}.png").astype(np.float32)[::-1].copy() / 65535.0 for v in ("ap", "lat")] for p in ids}  # ap (z,x), lat (z,y)
CT = {p: rd(ROOT / p / "ct.nii.gz").astype(np.int16) for p in ids}
def expert(pid):                                   # voxel (x, y, z)
    lm = json.load(open(ROOT / pid / "landmarks.json"))["landmarks"]
    return np.array(lm["femurHead"]["voxel"], float), np.array(lm["kneeCenter"]["voxel"], float)
print(len(ids), "patients | AP", DRR[ids[0]][0].shape, "LAT", DRR[ids[0]][1].shape, "| VSD:", VSDIN, "| planes:", PLANES)

In [ ]:
# data: same projection, pose and intensity augmentation as configuration C (train_v3)
def project(hu):
    mu = np.clip(0.02 * (1 + hu / 1000.0), 0, None)
    return [np.clip(mu.sum(axis=a) * VOX / DRR_SCALE, 0, 1).astype(np.float32) for a in (1, 2)]
def rot(rx, ry, rz):
    a, b, c = np.radians([rx, ry, rz])
    Rx = np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])
    Ry = np.array([[np.cos(b), 0, np.sin(b)], [0, 1, 0], [-np.sin(b), 0, np.cos(b)]])
    Rz = np.array([[np.cos(c), -np.sin(c), 0], [np.sin(c), np.cos(c), 0], [0, 0, 1]])
    return Rz @ Ry @ Rx
def pose(pid, rng):
    R = rot(rng.uniform(-15, 15), rng.uniform(-10, 10), rng.uniform(-15, 15))
    P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]]); M = (P @ R @ P).T     # output -> input, array order (z, y, x)
    c = (np.array(FULL) - 1) / 2; t = rng.uniform(-4, 4, 3); off = c - M @ (c + t)
    hu = ndimage.affine_transform(CT[pid].astype(np.float32), M, off, order=1, cval=-1024.0)
    lms = [(M.T @ (l[::-1] - off))[::-1] for l in expert(pid)]              # input = M out + off  ->  out = M^T (input - off)
    return (*project(hu), lms)
def aug_img(im, rng):
    im = np.clip(im, 0, 1) ** rng.uniform(0.6, 1.6)
    im = im * rng.uniform(0.7, 1.3) + rng.uniform(-0.1, 0.1)
    if rng.random() < 0.3: im = ndimage.gaussian_filter(im, rng.uniform(0.3, 1.5))
    if rng.random() < 0.5: im = im + rng.normal(0, rng.uniform(0, 0.04), im.shape)
    return im.astype(np.float32)
norm = lambda im: ((im - im.mean()) / (im.std() + 1e-6)).astype(np.float32)
def heat(shape, centres, s=2.0):                   # one Gaussian per landmark, centres as (row, col)
    r, c = np.arange(shape[0])[:, None], np.arange(shape[1])[None]
    return np.stack([np.exp(-((r - a) ** 2 + (c - b) ** 2) / (2 * s * s)) for a, b in centres]).astype(np.float32)
inside = lambda l: 0 <= l[0] < FULL[2] and 0 <= l[1] < FULL[1] and 0 <= l[2] < FULL[0]
def targets(ap, lat, lms):                         # AP pixel = (z, x), LAT pixel = (z, y)
    return heat(ap.shape, [(l[2], l[0]) for l in lms]), heat(lat.shape, [(l[2], l[1]) for l in lms])

class Train(Dataset):
    def __init__(self, ids_): self.ids = ids_
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        rng = np.random.default_rng(); pid = self.ids[i]
        ap, lat, lms = pose(pid, rng) if rng.random() < 0.8 else (*DRR[pid], list(expert(pid)))
        if not all(inside(l) for l in lms): ap, lat, lms = (*DRR[pid], list(expert(pid)))
        ha, hl = targets(ap, lat, lms)
        t = lambda a: torch.from_numpy(a)
        return t(norm(aug_img(ap, rng))[None]), t(norm(aug_img(lat, rng))[None]), t(ha), t(hl)

# check: one augmented sample with its target landmarks
ap, lat, ha, hl = Train(ids[:1])[0]
fig, ax = plt.subplots(1, 2, figsize=(6, 6))
for a, im, h in zip(ax, (ap, lat), (ha, hl)):
    a.imshow(im[0], cmap="gray", origin="lower"); a.contour(h.sum(0), levels=[0.5], colors=["r"]); a.set_title("target hip + knee (red)")
plt.show()

In [ ]:
# model, loss, decoding, triangulation
make = lambda: UNet(spatial_dims=2, in_channels=1, out_channels=2, channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2), num_res_units=2).to(device)
def hm_loss(logits, target):                       # cross-entropy between the spatial softmax and the normalised Gaussian
    B, C = logits.shape[:2]
    lp = F.log_softmax(logits.reshape(B, C, -1), -1); t = target.reshape(B, C, -1)
    return -((t / t.sum(-1, keepdim=True)) * lp).sum(-1).mean()
def decode(prob, r=3):                             # argmax, then probability-weighted centroid in a small window
    out = []
    for p in prob:
        a, b = np.unravel_index(p.argmax(), p.shape)
        a0, a1, b0, b1 = max(a - r, 0), min(a + r + 1, p.shape[0]), max(b - r, 0), min(b + r + 1, p.shape[1])
        w = p[a0:a1, b0:b1]; rr, cc = np.mgrid[a0:a1, b0:b1]
        out.append(((w * rr).sum() / w.sum(), (w * cc).sum() / w.sum()))
    return out
def probs(nets, im):
    x = torch.from_numpy(norm(im))[None, None].to(device)
    with torch.no_grad():
        ps = [F.softmax(n(x).flatten(2), -1).reshape(1, 2, *im.shape) for n in nets]
    return torch.stack(ps).mean(0)[0].cpu().numpy()
def landmarks3d(ap, lat, nets_ap, nets_lat):       # returns hip, knee as voxel (x, y, z) + z disagreement between views
    (hza, hx), (kza, kx) = decode(probs(nets_ap, ap))
    (hzl, hy), (kzl, ky) = decode(probs(nets_lat, lat))
    return np.array([hx, hy, (hza + hzl) / 2]), np.array([kx, ky, (kza + kzl) / 2]), abs(hza - hzl) * VOX, abs(kza - kzl) * VOX
def load(f):
    na, nl = make(), make(); s = torch.load(OUT / f"lm2d_refit_fold{f}.pt", map_location=device)
    na.load_state_dict(s["ap"]); nl.load_state_dict(s["lat"]); return na.eval(), nl.eval()

In [ ]:
# training: epochs per fold from the nested run, then retrain on all training patients, no selection
EPOCHS = 150                                       # cosine T_max as in the nested run, so the LR follows the same curve
N_EP = {f: int(torch.load(NESTED / f"lm2d_fold{f}.pt", map_location="cpu")["epoch"]) for f in range(5)}
print("epochs per fold (from nested inner validation):", N_EP)
for f in range(5):
    if (OUT / f"lm2d_refit_fold{f}.pt").exists(): print("fold", f, "done"); continue
    torch.manual_seed(42 + f); t0 = time.time()
    tr = [p for k, fl in enumerate(folds) if k != f for p in fl]
    print(f"fold {f}: train {len(tr)} (all training folds), {N_EP[f]} epochs, outer test {len(folds[f])}")
    na, nl = make(), make()
    opt = torch.optim.AdamW(list(na.parameters()) + list(nl.parameters()), lr=1e-3, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    dl = DataLoader(Train(tr), batch_size=4, shuffle=True, num_workers=4, persistent_workers=True, drop_last=True)
    for ep in range(N_EP[f]):
        na.train(); nl.train(); tl = 0
        for ap, lat, ha, hl in dl:
            loss = hm_loss(na(ap.to(device)), ha.to(device)) + hm_loss(nl(lat.to(device)), hl.to(device))
            opt.zero_grad(); loss.backward(); opt.step(); tl += loss.item()
        sched.step()
        if ep % 10 == 0 or ep == N_EP[f] - 1: print(f"fold {f} ep {ep+1:3d}/{N_EP[f]} loss {tl/len(dl):.3f}", flush=True)
    torch.save({"ap": na.state_dict(), "lat": nl.state_dict(), "epoch": N_EP[f]}, OUT / f"lm2d_refit_fold{f}.pt")
    print(f"fold {f} finished, {(time.time()-t0)/60:.0f} min", flush=True)


In [ ]:
# evaluation on HFValid (each patient by the fold model that held it out) + comparison with the 3D models
ang3 = lambda a, b: float(np.degrees(np.arccos(np.clip(a @ b / np.linalg.norm(a) / np.linalg.norm(b), -1, 1))))
cor = lambda v: float(np.degrees(np.arctan2(-v[0], v[2]))); sag = lambda v: float(np.degrees(np.arctan2(v[1], v[2])))
def errs(h, k, eh, ek):
    v, ve = h - k, eh - ek
    return dict(hip_mm=np.linalg.norm(h - eh) * VOX, knee_mm=np.linalg.norm(k - ek) * VOX, angle_deg=ang3(v, ve),
                coronal=abs(cor(v) - cor(ve)), sagittal=abs(sag(v) - sag(ve)))
NETS = {f: load(f) for f in range(5)}
rows = []
for p in ids:
    na, nl = NETS[FOLD[p]]; h, k, dzh, dzk = landmarks3d(*DRR[p], [na], [nl]); eh, ek = expert(p)
    r = dict(model="lm2d", patient=p, **errs(h, k, eh, ek), zdiff_hip_mm=dzh, zdiff_knee_mm=dzk,
             **{f"{n}_{a}": v * VOX for n, vec in (("h", h), ("k", k), ("eh", eh), ("ek", ek)) for a, v in zip("xyz", vec)}); r["fail"] = r["hip_mm"] > 20; rows.append(r)
L = pd.DataFrame(rows); L.to_csv(OUT / "lm2d_hfvalid_cv.csv", index=False)
CV = pd.concat([pd.read_csv(RES / "v3_hfvalid_cv.csv"), L], ignore_index=True)
CV["fail"] = CV.fail.astype(str) == "True"
q = lambda s: f"{s.median():.2f} [{s.quantile(.25):.2f}, {s.quantile(.75):.2f}]"
print("HFValid, median [IQR], n = 101")
for m in ["lm2d", "attnC", "swinC"]:
    g = CV[CV.model == m]
    print(f"  {m:6} axis {q(g.angle_deg)}  hip {q(g.hip_mm)}  knee {q(g.knee_mm)}  <1/<2 deg {(g.angle_deg<1).mean():.0%}/{(g.angle_deg<2).mean():.0%}"
          f"  worst {g.angle_deg.max():.2f}  failures {int(g.fail.sum())}")
g = L; print(f"  lm2d coronal {q(g.coronal)}  sagittal {q(g.sagittal)} | view z-disagreement hip {q(g.zdiff_hip_mm)} mm, knee {q(g.zdiff_knee_mm)} mm")
if PLANES is not None:
    PE = pd.read_csv(PLANES / "plane_errors.csv"); PE = PE[PE.cohort == "HFValid"]
    for m in ["attnC", "swinC"]:
        s = PE[PE.model == m]; print(f"  {m} coronal {q(s.coronal.abs())}  sagittal {q(s.sagittal.abs())}")

def holm(ps):
    o = np.argsort(ps); adj = np.empty(len(ps)); run = 0
    for i, j in enumerate(o): run = max(run, min(1, (len(ps) - i) * ps[j])); adj[j] = run
    return adj
rng = np.random.default_rng(0)
def compare(df, a, b, key, metrics):
    A, B = df[df.model == a].set_index(key), df[df.model == b].set_index(key)
    ds = {m: (A[m] - B[m].reindex(A.index)).dropna() for m in metrics}
    ps = holm([wilcoxon(d).pvalue for d in ds.values()])
    print(f"  {a} - {b}:")
    for (m, d), p in zip(ds.items(), ps):
        bs = [np.median(d.values[rng.integers(0, len(d), len(d))]) for _ in range(10000)]
        print(f"     {m:10} median diff {np.median(d):+.3f}  95% CI [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]  p_Holm {p:.3g}  (n={len(d)})")
    fa, fb = A.fail.astype(bool), B.fail.astype(bool).reindex(A.index); n10, n01 = int((fa & ~fb).sum()), int((~fa & fb).sum())
    print(f"     failures {int(fa.sum())} vs {int(fb.sum())}: McNemar p = {binomtest(n10, n10 + n01).pvalue if n10 + n01 else 1:.3g}")
print("\npaired comparisons (Wilcoxon, Holm over the 3 metrics; bootstrap CI of the median paired difference)")
for b in ["attnC", "swinC"]: compare(CV, "lm2d", b, "patient", ["angle_deg", "hip_mm", "knee_mm"])

In [ ]:
# error geometry for the 2D row (same definitions as the axis-decomposition notebook), tail vs expert p95
V = lambda n: L[[f"{n}_{a}" for a in "xyz"]].values
H, K, EH, EK = V("h"), V("k"), V("eh"), V("ek")
ax_ = EH - EK; u = ax_ / np.linalg.norm(ax_, axis=1, keepdims=True); flen = np.linalg.norm(ax_, axis=1)
along = lambda e: np.abs((e * u).sum(1)); acrossv = lambda e: e - (e * u).sum(1, keepdims=True) * u
dh, dk = H - EH, K - EK; ah, ak = acrossv(dh), acrossv(dk); ad = np.linalg.norm(ah - ak, axis=1)
approx = np.degrees(ad / flen)
print(f"2D  hip across {np.median(np.linalg.norm(ah, axis=1)):.2f} along {np.median(along(dh)):.2f} | knee across "
      f"{np.median(np.linalg.norm(ak, axis=1)):.2f} along {np.median(along(dk)):.2f} | across(hip-knee) {np.median(ad):.2f} "
      f"approx {np.median(approx):.2f} deg  axis {L.angle_deg.median():.2f} deg")
print(f"share of squared error across the axis: hip {(ah**2).sum()/(dh**2).sum():.2f}  knee {(ak**2).sum()/(dk**2).sum():.2f}")
Eh, Ek, Cr = (ah**2).sum(1).mean(), (ak**2).sum(1).mean(), -2 * (ah * ak).sum(1).mean(); T = Eh + Ek + Cr
print(f"mean-square split of across(hip-knee): hip {Eh/T:.0%}  knee {Ek/T:.0%}  correlation term {Cr/T:.0%}")
nf = ~L.fail.values; Eh2, Ek2, Cr2 = (ah[nf]**2).sum(1).mean(), (ak[nf]**2).sum(1).mean(), -2 * (ah[nf] * ak[nf]).sum(1).mean(); T2 = Eh2 + Ek2 + Cr2
print(f"  without head failures ({nf.sum()}): hip {Eh2/T2:.0%}  knee {Ek2/T2:.0%}  correlation term {Cr2/T2:.0%}")
print(f"tail: axis > 0.81 deg (expert p95) in {(L.angle_deg > 0.81).mean():.0%} of cases")


In [ ]:
# external test on VSD: mean of the 5 fold models; axis vs experts on the 19 femurs, head failures over all 58
#   (failure reference = hip centre fitted on the true mask, exactly as for the 3D models)
def surface_points(m):
    z, y, x = np.nonzero(m & ~ndimage.binary_erosion(m)); return np.stack([x, y, z], 1).astype(float)
def _sphere_lsq(p):
    s, *_ = np.linalg.lstsq(np.c_[2 * p, np.ones(len(p))], (p ** 2).sum(1), rcond=None)
    c = s[:3]; r2 = s[3] + c @ c; return (c, np.sqrt(r2)) if r2 > 0 else (None, None)
def fit_head(mask, prox=0.35, med=0.6, r_vox=(9, 14), n_iter=500, seed=0):
    p = surface_points(mask); z = p[:, 2]; p = p[z >= z.max() - prox * (z.max() - z.min())]
    xl, xh = p[:, 0].min(), p[:, 0].max(); p = p[p[:, 0] <= xl + med * (xh - xl)]
    rng_ = np.random.default_rng(seed); bn, bc, br = -1, None, None
    for _ in range(n_iter):
        c, r = _sphere_lsq(p[rng_.choice(len(p), 4, replace=False)])
        if c is None or not (r_vox[0] <= r <= r_vox[1]): continue
        k = (np.abs(np.linalg.norm(p - c, axis=1) - r) < 2.0).sum()
        if k > bn: bn, bc, br = k, c, r
    for t in (2.0, 1.5, 1.0):
        inl = np.abs(np.linalg.norm(p - bc, axis=1) - br) < t; bc, br = _sphere_lsq(p[inl])
    return bc
def fill_solid(seg):
    s = ndimage.binary_fill_holes(seg); return np.stack([ndimage.binary_fill_holes(sl) for sl in s])
if VSDIN is None:
    print("VSD output not attached -> skipping")
else:
    VD = VSDIN / "data"; AP_N, LAT_N = [NETS[f][0] for f in range(5)], [NETS[f][1] for f in range(5)]
    rows = []
    for fid in sorted(p.name for p in VD.iterdir() if (p / "data.npz").exists()):
        d = np.load(VD / fid / "data.npz"); h, k, _, _ = landmarks3d(d["ap"].astype(np.float32), d["lat"].astype(np.float32), AP_N, LAT_N)
        r = dict(model="lm2d", femur=fid, hip_vs_gtmask_mm=np.linalg.norm(h - fit_head(fill_solid(d["seg"]))) * VOX)
        lp = VD / fid / "landmarks.json"
        if lp.exists():
            lm = json.load(open(lp)); r.update(errs(h, k, np.array(lm["hip"]), np.array(lm["knee"])))
        r["fail"] = r["hip_vs_gtmask_mm"] > 20; rows.append(r)
    LV = pd.DataFrame(rows); LV.to_csv(OUT / "lm2d_vsd.csv", index=False)
    VS = pd.concat([pd.read_csv(RES / "v3_vsd.csv"), LV], ignore_index=True); VS["fail"] = VS.fail.astype(str) == "True"
    print("VSD: axis/hip/knee vs experts (n=19), failures over 58")
    for m in ["lm2d", "attnC", "swinC"]:
        g = VS[VS.model == m]; e = g.dropna(subset=["angle_deg"])
        print(f"  {m:6} axis {q(e.angle_deg)}  hip {q(e.hip_mm)}  knee {q(e.knee_mm)}  axis<2 deg {(e.angle_deg<2).sum()}/{len(e)}  failures {int(g.fail.sum())}/{len(g)}")
    print(f"  lm2d coronal {q(LV.coronal.dropna())}  sagittal {q(LV.sagittal.dropna())}")
    E = VS.dropna(subset=["angle_deg"])
    for b in ["attnC", "swinC"]:
        compare(E, "lm2d", b, "femur", ["angle_deg", "hip_mm", "knee_mm"])
        A, B = VS[VS.model == "lm2d"].set_index("femur").fail, VS[VS.model == b].set_index("femur").fail
        B = B.reindex(A.index); n10, n01 = int((A & ~B).sum()), int((~A & B).sum())
        print(f"     failures over 58: {int(A.sum())} vs {int(B.sum())}, McNemar p = {binomtest(n10, n10 + n01).pvalue if n10 + n01 else 1:.3g}")

In [ ]:
# pictures: worst and median HFValid cases (predicted red x, expert cyan +)
L2 = L.sort_values("angle_deg"); show = [L2.patient.iloc[len(L2) // 2], L2.patient.iloc[-2], L2.patient.iloc[-1]]
fig, ax = plt.subplots(1, 6, figsize=(14, 6))
for i, p in enumerate(show):
    na, nl = NETS[FOLD[p]]; h, k, _, _ = landmarks3d(*DRR[p], [na], [nl]); eh, ek = expert(p)
    for j, (im, col) in enumerate(((DRR[p][0], 0), (DRR[p][1], 1))):
        a = ax[2 * i + j]; a.imshow(im, cmap="gray", origin="lower"); a.axis("off")
        for pt, mk, c in ((h, "x", "r"), (k, "x", "r"), (eh, "+", "c"), (ek, "+", "c")): a.plot(pt[col], pt[2], mk, c=c, ms=9, mew=2)
    ax[2 * i].set_title(f"{p}: axis {L.set_index('patient').angle_deg[p]:.2f} deg", fontsize=9)
plt.tight_layout(); fig.savefig(OUT / "lm2d_cases.png", dpi=150); plt.show()
shutil.make_archive("/kaggle/working/lm2d_refit", "zip", OUT); print("done -> download lm2d_refit.zip")